## ⚙️ Cellule 1 — Configuration

In [1]:
from google.colab import drive
import os, json

if not os.path.ismount('/content/drive'):
    drive.mount('/content/drive')
else:
    print('Drive déjà monté')

# ─── CHEMINS ────────────────────────────────────────────────────
FRACATLAS_ZIP    = '/content/drive/MyDrive/FracAtlas/archive.zip'
FRACATLAS_ROOT   = '/content/FracAtlas_extracted'
IMAGES_SUBDIRS   = ['Fractured', 'Non_fractured']
LABELS_DIR       = os.path.join(FRACATLAS_ROOT, 'Annotations', 'YOLO')
SPLITS_PATH      = '/content/drive/MyDrive/resultats_finaux_fracture/splits_fracatlas.json'
STATS_PATH       = '/content/drive/MyDrive/resultats_finaux_fracture/fracatlas_clahe_stats.json'
SAVE_DIR         = '/content/drive/MyDrive/resultats_finaux_fracture/YOLOv11n'  # CORRECTIF : nouveau dossier, distinct de l'ancien run (optimizer/lr0 non spécifiés) — on ne veut pas charger par erreur d'anciens checkpoints entraînés avec les valeurs par défaut d'ultralytics
YOLO_DATA        = '/content/yolo_fracatlas_dataset'
os.makedirs(SAVE_DIR, exist_ok=True)

# ─── CONFIG FINALE (post-pilote) ──────────────────────────────────
# 640 confirmé meilleur que 512 (mAP50 0.15 -> 0.33+ sur seed 42 pilote).
# epochs/patience augmentés pour laisser converger à cette résolution.
# Ces valeurs (weight_decay, degrees/translate/scale) doivent être répliquées
# à l'identique sur Faster R-CNN et RT-DETR pour garder le protocole harmonisé.
IMG_SIZE      = 640
EPOCHS        = 150
PATIENCE      = 20
WEIGHT_DECAY  = 0.0005
LR0           = 5e-4      # CORRECTIF : alignement explicite sur le protocole (§4.4), identique à Faster R-CNN. Non spécifié dans le run original -> valeurs par défaut ultralytics utilisées par erreur.
AUG_DEGREES   = 8.0
AUG_TRANSLATE = 0.1
AUG_SCALE     = 0.15

CLAHE_CLIP    = 2.0
CLAHE_GRID    = (8, 8)
LABEL_TO_NAME = {0: 'Fracture'}   # classe unique

# 5 seeds pour le volet fracture (vs 3 pour le thoracique)
SEEDS = [42, 123, 2024, 7, 77]

with open(SPLITS_PATH) as f:
    splits = json.load(f)
with open(STATS_PATH) as f:
    clahe_stats = json.load(f)

print(f"Train : {len(splits['train'])} images | Val : {len(splits['val'])} images | Test : {len(splits['test'])} images")
print(f"CLAHE stats (calculées sur {clahe_stats['n_images']} images train) : mean={clahe_stats['mean'][0]:.4f}, std={clahe_stats['std'][0]:.4f}")
print(f"\nIMG_SIZE={IMG_SIZE} | EPOCHS={EPOCHS} | PATIENCE={PATIENCE} | SEEDS={SEEDS}")

Mounted at /content/drive
Train : 2857 images | Val : 613 images | Test : 613 images
CLAHE stats (calculées sur 2857 images train) : mean=0.2116, std=0.0552

IMG_SIZE=640 | EPOCHS=150 | PATIENCE=20 | SEEDS=[42, 123, 2024, 7, 77]


## 📦 Cellule 1bis — Extraction de l'archive FracAtlas

L'archive n'a jamais été décompressée sur le Drive — on l'extrait une fois vers le disque local de la session Colab (plus rapide à lire pendant l'entraînement que depuis Drive), sans la ré-extraire si déjà fait dans la session en cours.

In [2]:
import zipfile

if os.path.isdir(FRACATLAS_ROOT) and len(os.listdir(FRACATLAS_ROOT)) > 0:
    print(f'✅ Déjà extrait dans cette session → {FRACATLAS_ROOT} ({len(os.listdir(FRACATLAS_ROOT))} éléments)')
else:
    assert os.path.exists(FRACATLAS_ZIP), f'Archive introuvable : {FRACATLAS_ZIP}'
    os.makedirs(FRACATLAS_ROOT, exist_ok=True)
    print(f'📦 Extraction de {FRACATLAS_ZIP} vers {FRACATLAS_ROOT}...')
    with zipfile.ZipFile(FRACATLAS_ZIP, 'r') as zf:
        zf.extractall(FRACATLAS_ROOT)
    print('✅ Extraction terminée.')

# Aperçu de l'arborescence extraite, pour confirmer/ajuster IMAGES_SUBDIRS et LABELS_DIR si besoin
print('\nContenu de premier niveau :', os.listdir(FRACATLAS_ROOT))
for root, dirs, files in os.walk(FRACATLAS_ROOT):
    depth = root[len(FRACATLAS_ROOT):].count(os.sep)
    if depth <= 2:
        print('  ' * depth + os.path.basename(root) + '/', f'({len(files)} fichiers)' if files else '')


📦 Extraction de /content/drive/MyDrive/FracAtlas/archive.zip vers /content/FracAtlas_extracted...
✅ Extraction terminée.

Contenu de premier niveau : ['FracAtlas']
FracAtlas_extracted/ 
  FracAtlas/ (1 fichiers)
    images/ 
    Utilities/ (2 fichiers)
    Annotations/ 


## 🔎 Cellule 2 — Vérification de la structure sur Drive

À lancer avant tout le reste. Confirme que les images et les labels sont bien là où on les cherche.

In [3]:
# Vérifie la présence des dossiers images et labels, et échantillonne quelques fichiers
for sub in IMAGES_SUBDIRS:
    d = os.path.join(FRACATLAS_ROOT, 'images', sub)
    status = '✅' if os.path.isdir(d) else '⚠️  introuvable'
    n = len(os.listdir(d)) if os.path.isdir(d) else 0
    print(f'{status}  images/{sub}  ({n} fichiers)  →  {d}')

status_lbl = '✅' if os.path.isdir(LABELS_DIR) else '⚠️  introuvable'
n_lbl = len(os.listdir(LABELS_DIR)) if os.path.isdir(LABELS_DIR) else 0
print(f'{status_lbl}  Annotations/YOLO  ({n_lbl} fichiers)  →  {LABELS_DIR}')

# Vérifie qu'un id du split train se retrouve bien quelque part (image + label)
sample_id = splits['train'][0]
stem = os.path.splitext(sample_id)[0]
found_img = None
for sub in IMAGES_SUBDIRS:
    p = os.path.join(FRACATLAS_ROOT, 'images', sub, sample_id)
    if os.path.exists(p):
        found_img = p
        break
found_lbl = os.path.join(LABELS_DIR, stem + '.txt')

print(f"\nTest sur '{sample_id}' :")
print(f'  image : {"✅ trouvée → " + found_img if found_img else "⚠️ NON trouvée dans " + str(IMAGES_SUBDIRS)}')
print(f'  label : {"✅ trouvé" if os.path.exists(found_lbl) else "⚠️ NON trouvé"} → {found_lbl}')
if os.path.exists(found_lbl):
    with open(found_lbl) as f:
        print(f'    contenu : {f.read().strip()}')
    print("    (format attendu : 'class_id x_center y_center width height', normalisé 0-1)")

print("\n⚠️ Si l'un des points ci-dessus échoue, corrige IMAGES_SUBDIRS / LABELS_DIR en Cellule 1 avant de continuer.")


⚠️  introuvable  images/Fractured  (0 fichiers)  →  /content/FracAtlas_extracted/images/Fractured
⚠️  introuvable  images/Non_fractured  (0 fichiers)  →  /content/FracAtlas_extracted/images/Non_fractured
⚠️  introuvable  Annotations/YOLO  (0 fichiers)  →  /content/FracAtlas_extracted/Annotations/YOLO

Test sur 'IMG0003173.jpg' :
  image : ⚠️ NON trouvée dans ['Fractured', 'Non_fractured']
  label : ⚠️ NON trouvé → /content/FracAtlas_extracted/Annotations/YOLO/IMG0003173.txt

⚠️ Si l'un des points ci-dessus échoue, corrige IMAGES_SUBDIRS / LABELS_DIR en Cellule 1 avant de continuer.


## 🔧 Cellule 3 — Prétraitement CLAHE + Letterbox + réécriture des labels YOLO

Important : le letterbox modifie les dimensions de l'image (redimensionnement + padding), donc les
coordonnées des boîtes doivent être recalculées — pas seulement recopiées. On part des coordonnées
YOLO normalisées d'origine (relatives à l'image brute), on les repasse en pixels absolus, on applique
le même facteur d'échelle et le même décalage de padding que l'image, puis on renormalise par `IMG_SIZE`.

In [4]:
import cv2
import numpy as np
from tqdm.auto import tqdm

# ── Index robuste des images : balayage complet de FRACATLAS_ROOT ──
# Plutôt que de supposer une arborescence fixe (images/Fractured, images/Non_fractured...),
# on indexe UNE FOIS tous les fichiers image trouvés sous FRACATLAS_ROOT, par nom de fichier.
# Ça évite de dépendre d'une structure de dossiers qu'on ne connaît pas avec certitude.
IMG_EXTENSIONS = {'.jpg', '.jpeg', '.png', '.bmp', '.tif', '.tiff'}

print(f'🔎 Indexation des images sous {FRACATLAS_ROOT}...')
_image_index = {}
_n_walked = 0
for root, dirs, files in os.walk(FRACATLAS_ROOT):
    for fname in files:
        if os.path.splitext(fname)[1].lower() in IMG_EXTENSIONS:
            _image_index[fname] = os.path.join(root, fname)
            _n_walked += 1
print(f'✅ {_n_walked} fichiers image indexés sous {FRACATLAS_ROOT}')

# Vérification immédiate sur un échantillon du split train (fail-fast si toujours 0 match)
_sample = splits['train'][:5]
_found = [i for i in _sample if i in _image_index]
print(f'   Échantillon train : {len(_found)}/{len(_sample)} trouvés → {_found}')
if len(_found) == 0:
    raise RuntimeError(
        f"Aucune image du split train n'a été retrouvée sous {FRACATLAS_ROOT}. "
        f"Vérifie FRACATLAS_ROOT en Cellule 1 (le dossier racine est-il le bon ?), "
        f"ou que le nom exact des fichiers (ex: '{_sample[0]}') correspond bien à ce qui est sur le Drive."
    )

def find_image_path(image_id):
    return _image_index.get(image_id, None)

# ── CORRECTIF : index robuste des LABELS, même logique que pour les images ──
# LABELS_DIR (Cellule 1) est un chemin statique qui suppose FRACATLAS_ROOT/Annotations/YOLO.
# Si le zip extrait une arborescence imbriquée (ex: FRACATLAS_ROOT/FracAtlas/Annotations/YOLO),
# ce chemin statique ne pointe vers rien, et l'ancien code traitait ça silencieusement comme
# "pas de label" pour TOUTES les images (bug identifié : 100% des images en background).
# On corrige en indexant tous les .txt trouvés sous FRACATLAS_ROOT par nom de fichier,
# restreint aux chemins contenant 'Annotations' pour éviter de capter un .txt parasite ailleurs.
print(f'🔎 Indexation des labels YOLO sous {FRACATLAS_ROOT}...')
_label_index = {}
for root, dirs, files in os.walk(FRACATLAS_ROOT):
    if 'Annotations' not in root:
        continue
    for fname in files:
        if fname.endswith('.txt'):
            _label_index[fname] = os.path.join(root, fname)
print(f'✅ {len(_label_index)} fichiers .txt indexés sous Annotations/ (attendu : ~4083)')

def find_label_path(stem):
    return _label_index.get(stem + '.txt', None)

# Vérification fail-fast : on cherche spécifiquement des images FRACTURÉES du train
# (pas juste les 5 premières, qui sont probablement non-fracturées et auraient un label
# vide de toute façon — ça ne prouverait rien sur le bug qu'on corrige ici)
_fractured_sample = [
    i for i in splits['train']
    if find_image_path(i) and 'Fractured' in find_image_path(i) and 'Non_fractured' not in find_image_path(i)
][:5]
_labels_found = 0
for _fid in _fractured_sample:
    _stem = os.path.splitext(_fid)[0]
    _lp = find_label_path(_stem)
    if _lp and os.path.getsize(_lp) > 0:
        _labels_found += 1
print(f'   Échantillon train FRACTURÉ : {_labels_found}/{len(_fractured_sample)} ont un label non-vide '
      f'{_fractured_sample}')
if len(_fractured_sample) > 0 and _labels_found == 0:
    raise RuntimeError(
        "Aucun label non-vide trouvé pour des images fracturées du train. "
        "Le chemin Annotations/YOLO est peut-être introuvable dans l'arborescence extraite, "
        "ou le nommage des fichiers .txt ne correspond pas aux noms d'images. "
        "Vérifie la sortie de l'indexation ci-dessus (nombre de .txt trouvés)."
    )

def apply_clahe(img_gray):
    clahe = cv2.createCLAHE(clipLimit=CLAHE_CLIP, tileGridSize=CLAHE_GRID)
    return clahe.apply(img_gray)

def letterbox_image(img, size=IMG_SIZE):
    """Redimensionne en conservant le ratio + padding centré. Retourne l'image + les paramètres
    de transformation (scale, pad_x, pad_y) nécessaires pour ajuster les boîtes."""
    h, w = img.shape[:2]
    scale = min(size / h, size / w)
    new_h, new_w = int(round(h * scale)), int(round(w * scale))
    resized = cv2.resize(img, (new_w, new_h), interpolation=cv2.INTER_AREA)
    pad_y, pad_x = (size - new_h) // 2, (size - new_w) // 2
    canvas = np.full((size, size), 114, dtype=np.uint8)  # gris neutre, convention YOLO
    canvas[pad_y:pad_y + new_h, pad_x:pad_x + new_w] = resized
    return canvas, scale, pad_x, pad_y

def letterbox_yolo_boxes(label_path, orig_w, orig_h, scale, pad_x, pad_y, size=IMG_SIZE):
    """Relit un fichier de labels YOLO (coords normalisées / image d'origine) et renvoie les
    lignes YOLO recalculées pour l'image letterboxée."""
    if label_path is None or not os.path.exists(label_path):
        return []  # image sans fracture annotée (ne devrait pas arriver si Fractured/ contient une box)
    new_lines = []
    with open(label_path) as f:
        for line in f:
            parts = line.strip().split()
            if len(parts) != 5:
                continue
            cls, xc, yc, bw, bh = parts
            xc, yc, bw, bh = float(xc), float(yc), float(bw), float(bh)
            # normalisé -> pixels absolus (image d'origine)
            xc_px, yc_px = xc * orig_w, yc * orig_h
            bw_px, bh_px = bw * orig_w, bh * orig_h
            # applique le même scale + padding que l'image
            xc_new = xc_px * scale + pad_x
            yc_new = yc_px * scale + pad_y
            bw_new = bw_px * scale
            bh_new = bh_px * scale
            # renormalise par IMG_SIZE
            new_lines.append(f'{cls} {xc_new/size:.6f} {yc_new/size:.6f} {bw_new/size:.6f} {bh_new/size:.6f}')
    return new_lines

def preprocess_split(image_ids, split_name):
    img_out_dir = os.path.join(YOLO_DATA, 'images', split_name)
    lbl_out_dir = os.path.join(YOLO_DATA, 'labels', split_name)
    os.makedirs(img_out_dir, exist_ok=True)
    os.makedirs(lbl_out_dir, exist_ok=True)

    n_missing_img, n_no_label, n_with_label = 0, 0, 0
    for image_id in tqdm(image_ids, desc=f'Prétraitement {split_name}'):
        stem = os.path.splitext(image_id)[0]
        src_path = find_image_path(image_id)
        if src_path is None:
            n_missing_img += 1
            continue

        img = cv2.imread(src_path, cv2.IMREAD_GRAYSCALE)
        orig_h, orig_w = img.shape[:2]
        img_clahe = apply_clahe(img)
        img_lb, scale, pad_x, pad_y = letterbox_image(img_clahe, IMG_SIZE)
        img_rgb = cv2.cvtColor(img_lb, cv2.COLOR_GRAY2BGR)  # ultralytics attend du 3-canaux
        cv2.imwrite(os.path.join(img_out_dir, stem + '.jpg'), img_rgb)

        label_path = find_label_path(stem)  # CORRECTIF : index robuste au lieu du chemin statique
        new_lines = letterbox_yolo_boxes(label_path, orig_w, orig_h, scale, pad_x, pad_y, IMG_SIZE)
        if label_path is None:
            n_no_label += 1
        elif new_lines:
            n_with_label += 1
        with open(os.path.join(lbl_out_dir, stem + '.txt'), 'w') as f:
            f.write('\n'.join(new_lines))  # vide = image négative (Non_fractured), convention YOLO

    print(f'  ✅ {split_name} : {len(image_ids) - n_missing_img}/{len(image_ids)} images traitées '
          f'({n_missing_img} introuvables, {n_no_label} sans fichier .txt trouvé, '
          f'{n_with_label} avec au moins une bbox)')

for split_name, ids in [('train', splits['train']), ('val', splits['val']), ('test', splits['test'])]:
    preprocess_split(ids, split_name)


🔎 Indexation des images sous /content/FracAtlas_extracted...
✅ 4083 fichiers image indexés sous /content/FracAtlas_extracted
   Échantillon train : 5/5 trouvés → ['IMG0003173.jpg', 'IMG0000063.jpg', 'IMG0002055.jpg', 'IMG0002154.jpg', 'IMG0001769.jpg']
🔎 Indexation des labels YOLO sous /content/FracAtlas_extracted...
✅ 4084 fichiers .txt indexés sous Annotations/ (attendu : ~4083)
   Échantillon train FRACTURÉ : 5/5 ont un label non-vide ['IMG0002607.jpg', 'IMG0002483.jpg', 'IMG0002577.jpg', 'IMG0002127.jpg', 'IMG0001494.jpg']


Prétraitement train:   0%|          | 0/2857 [00:00<?, ?it/s]

  ✅ train : 2857/2857 images traitées (0 introuvables, 0 sans fichier .txt trouvé, 513 avec au moins une bbox)


Prétraitement val:   0%|          | 0/613 [00:00<?, ?it/s]

  ✅ val : 613/613 images traitées (0 introuvables, 0 sans fichier .txt trouvé, 96 avec au moins une bbox)


Prétraitement test:   0%|          | 0/613 [00:00<?, ?it/s]

  ✅ test : 613/613 images traitées (0 introuvables, 0 sans fichier .txt trouvé, 108 avec au moins une bbox)


## ⚖️ Cellule 3bis — Suréchantillonnage des images fracturées (train uniquement)

Objectif : rapprocher le mAP50 de la fourchette littérature, en augmentant la proportion d'images
positives vues pendant l'entraînement. Val/test restent inchangés (distribution naturelle) pour
garder une évaluation honnête.

Méthode : duplication physique des paires image/label des images fracturées du train (pas de
`WeightedRandomSampler` natif dans le pipeline ultralytics basé dossiers). Chaque copie est revue
par l'augmentation géométrique d'ultralytics à chaque epoch (rotation/translation/scale déjà
configurées en Cellule 1), donc les duplicatas ne sont pas des images strictement identiques
d'un epoch à l'autre.

Cible retenue : **~35-40% de positifs dans le train** (choix modéré plutôt que les ~54% de
l'étude de référence, pour limiter le risque de sur-apprentissage sur les ~500 images
positives sources — leur résultat combine aussi une restriction aux radios de la main, absente
ici). À répliquer sur Faster R-CNN / RT-DETR (via `WeightedRandomSampler`) pour rester harmonisé
si le gain se confirme.

In [5]:
import shutil

TARGET_POSITIVE_RATIO = 0.37  # ~35-40%, choix validé (cf. cellule markdown ci-dessus)

train_img_dir = os.path.join(YOLO_DATA, 'images', 'train')
train_lbl_dir = os.path.join(YOLO_DATA, 'labels', 'train')

# Identifie les images positives (label non vide) vs négatives dans le train prétraité
all_stems = [os.path.splitext(f)[0] for f in os.listdir(train_img_dir)]
pos_stems, neg_stems = [], []
for stem in all_stems:
    lbl_path = os.path.join(train_lbl_dir, stem + '.txt')
    if os.path.exists(lbl_path) and os.path.getsize(lbl_path) > 0:
        pos_stems.append(stem)
    else:
        neg_stems.append(stem)

n_pos, n_neg = len(pos_stems), len(neg_stems)
current_ratio = n_pos / (n_pos + n_neg)
print(f'Avant suréchantillonnage : {n_pos} positifs / {n_pos + n_neg} total ({current_ratio:.1%})')

# k = nombre total de copies par image positive (originale incluse) pour atteindre TARGET_POSITIVE_RATIO
# dérivé de : n_pos*k / (n_pos*k + n_neg) = TARGET_POSITIVE_RATIO
k = (TARGET_POSITIVE_RATIO * n_neg) / (n_pos * (1 - TARGET_POSITIVE_RATIO))
k_extra = max(0, round(k) - 1)  # nombre de duplicatas à ajouter par image positive
print(f'Facteur de duplication calculé : k={k:.2f} → {k_extra} copie(s) supplémentaire(s) par image positive')

n_created = 0
for stem in pos_stems:
    src_img = os.path.join(train_img_dir, stem + '.jpg')
    src_lbl = os.path.join(train_lbl_dir, stem + '.txt')
    for dup_i in range(1, k_extra + 1):
        dst_img = os.path.join(train_img_dir, f'{stem}_dup{dup_i}.jpg')
        dst_lbl = os.path.join(train_lbl_dir, f'{stem}_dup{dup_i}.txt')
        if not os.path.exists(dst_img):
            shutil.copyfile(src_img, dst_img)
            shutil.copyfile(src_lbl, dst_lbl)
            n_created += 1

n_pos_after = n_pos * (1 + k_extra)
n_total_after = n_pos_after + n_neg
print(f'✅ {n_created} paires image/label créées par duplication (train uniquement)')
print(f'Après suréchantillonnage : {n_pos_after} positifs / {n_total_after} total ({n_pos_after / n_total_after:.1%})')
print('⚠️ val/test non touchés — distribution naturelle conservée pour une évaluation honnête.')

Avant suréchantillonnage : 513 positifs / 2857 total (18.0%)
Facteur de duplication calculé : k=2.68 → 2 copie(s) supplémentaire(s) par image positive
✅ 1026 paires image/label créées par duplication (train uniquement)
Après suréchantillonnage : 1539 positifs / 3883 total (39.6%)
⚠️ val/test non touchés — distribution naturelle conservée pour une évaluation honnête.


## 📦 Cellule 4 — data.yaml + installation ultralytics

In [6]:
import subprocess, sys
# Version épinglée : les installations 'ultralytics' les plus récentes ont pu introduire
# une génération de modèles plus récente (YOLO26) qui court-circuite 'yolo11n.pt' au
# téléchargement (observé : yolo26n.pt téléchargé à la place). On fixe une version connue
# pour garantir que YOLOv11 est bien ce qui est chargé.
subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', 'ultralytics==8.3.40'], check=True)

data_yaml_path = os.path.join(YOLO_DATA, 'data.yaml')
data_yaml_content = f"""path: {YOLO_DATA}
train: images/train
val: images/val
test: images/test
nc: 1
names: ['Fracture']
"""
with open(data_yaml_path, 'w') as f:
    f.write(data_yaml_content)

print('✅ ultralytics installé, data.yaml écrit :')
print(data_yaml_content)


✅ ultralytics installé, data.yaml écrit :
path: /content/yolo_fracatlas_dataset
train: images/train
val: images/val
test: images/test
nc: 1
names: ['Fracture']



## 🚀 Cellule 5 — Entraînement multi-seed YOLOv11n

Checkpoints repris depuis le disque si déjà entraînés (même convention que les autres notebooks).

In [7]:
from ultralytics import YOLO
import shutil, torch, gc

trained_weights = {}  # seed -> chemin vers best.pt

for seed in SEEDS:
    seed_dir = os.path.join(SAVE_DIR, f'seed_{seed}')
    best_ckpt = os.path.join(seed_dir, 'weights', 'best.pt')

    if os.path.exists(best_ckpt):
        print(f'✅ Seed {seed} : checkpoint déjà entraîné, chargement → {best_ckpt}')
        trained_weights[seed] = best_ckpt
        continue

    print(f'\n🚀 Seed {seed} — entraînement YOLOv11n (config CORRIGÉE : SGD, lr0={LR0}, 640px)...')
    model = YOLO('yolo11n.pt')
    assert 'yolo11' in str(model.ckpt_path).lower() or 'v11' in str(getattr(model, 'model_name', '')).lower(), (
        f"⚠️  Le modèle chargé ne semble pas être YOLOv11 (ckpt: {model.ckpt_path}) — "
        f"vérifier la version d'ultralytics installée."
    )
    model.train(
        data=data_yaml_path,
        imgsz=IMG_SIZE,
        epochs=EPOCHS,
        patience=PATIENCE,
        batch=16,
        seed=seed,
        project=SAVE_DIR,
        name=f'seed_{seed}',
        exist_ok=True,
        # CORRECTIF : optimiseur et LR explicites, alignés sur Faster R-CNN (§4.4 du protocole).
        # Absents du run original -> ultralytics appliquait alors ses valeurs par défaut
        # (optimizer='auto', lr0=0.01), ce qui cassait la comparabilité inter-architectures.
        optimizer='SGD',
        lr0=LR0,
        momentum=0.9,             # aligné sur Faster R-CNN / RT-DETR (§4.4)
        # Régularisation portable (réplicable à l'identique sur Faster R-CNN / RT-DETR)
        weight_decay=WEIGHT_DECAY,
        # CLAHE gère déjà le contraste -> pas d'augmentation couleur native
        hsv_h=0.0, hsv_s=0.0, hsv_v=0.0,
        # Augmentation géométrique modérée, portable (transformations affines génériques)
        degrees=AUG_DEGREES, translate=AUG_TRANSLATE, scale=AUG_SCALE,
        shear=0.0, perspective=0.0,
        flipud=0.0,  # retiré : orientation anatomique à valider avant réactivation
        fliplr=0.5,
        # Désactivés : spécifiques à l'implémentation ultralytics, non réplicables
        # à l'identique sur Faster R-CNN (torchvision) ni RT-DETR
        mosaic=0.0, mixup=0.0,
    )
    trained_weights[seed] = best_ckpt
    print(f'✅ Seed {seed} — terminé, meilleur checkpoint → {best_ckpt}')

    del model
    gc.collect()
    if torch.cuda.is_available():
        torch.cuda.empty_cache()

print(f'\n🏁 Entraînement terminé pour {len(SEEDS)} seed(s) : {SEEDS}.')

Creating new Ultralytics Settings v0.0.6 file ✅ 
View Ultralytics Settings with 'yolo settings' or at '/root/.config/Ultralytics/settings.json'
Update Settings with 'yolo settings key=value', i.e. 'yolo settings runs_dir=path/to/dir'. For help see https://docs.ultralytics.com/quickstart/#ultralytics-settings.
✅ Seed 42 : checkpoint déjà entraîné, chargement → /content/drive/MyDrive/resultats_finaux_fracture/YOLOv11n/seed_42/weights/best.pt
✅ Seed 123 : checkpoint déjà entraîné, chargement → /content/drive/MyDrive/resultats_finaux_fracture/YOLOv11n/seed_123/weights/best.pt
✅ Seed 2024 : checkpoint déjà entraîné, chargement → /content/drive/MyDrive/resultats_finaux_fracture/YOLOv11n/seed_2024/weights/best.pt
✅ Seed 7 : checkpoint déjà entraîné, chargement → /content/drive/MyDrive/resultats_finaux_fracture/YOLOv11n/seed_7/weights/best.pt
✅ Seed 77 : checkpoint déjà entraîné, chargement → /content/drive/MyDrive/resultats_finaux_fracture/YOLOv11n/seed_77/weights/best.pt

🏁 Entraînement term

## 📊 Cellule 6 — Évaluation multi-seed (AUC, sensibilité, spécificité, mAP50)

Sur le test set. Classe unique ('Fracture'), donc AUC/sensibilité/spécificité au niveau image
(présence ou non d'au moins une fracture détectée), + mAP50 pour la qualité de localisation.

In [8]:
from ultralytics.utils.metrics import ap_per_class
from sklearn.metrics import roc_auc_score, roc_curve
import numpy as np

results_by_seed = {}

for seed in SEEDS:
    metrics_path = os.path.join(SAVE_DIR, f'metriques_seed{seed}.json')

    if os.path.exists(metrics_path):
        with open(metrics_path) as f:
            cached = json.load(f)
        if 'mAP50' in cached.get('Fracture', {}):
            results_by_seed[seed] = cached
            print(f'✅ Seed {seed} : métriques déjà calculées, chargées depuis le disque.')
            continue

    print(f'\n📊 Seed {seed} — évaluation...')
    model = YOLO(trained_weights[seed])

    # mAP50 (via la validation native ultralytics, IoU=0.5 uniquement pour cohérence
    # avec le protocole thoracique — cf. correctif Faster R-CNN/RetinaNet)
    val_results = model.val(data=data_yaml_path, split='test', imgsz=IMG_SIZE, iou=0.5, conf=0.001)
    map50 = float(val_results.box.map50)

    # Score image-level (max de confiance des boîtes prédites, 0 si aucune détection)
    # pour l'AUC/sensibilité/spécificité, en miroir des autres modèles du protocole.
    test_dir = os.path.join(YOLO_DATA, 'images', 'test')
    labels_dir = os.path.join(YOLO_DATA, 'labels', 'test')
    y_true, y_score = [], []
    for fname in sorted(os.listdir(test_dir)):
        stem = os.path.splitext(fname)[0]
        lbl_path = os.path.join(labels_dir, stem + '.txt')
        has_fracture = os.path.exists(lbl_path) and os.path.getsize(lbl_path) > 0
        pred = model.predict(os.path.join(test_dir, fname), imgsz=IMG_SIZE, conf=0.001, verbose=False)[0]
        score = float(pred.boxes.conf.max()) if len(pred.boxes) > 0 else 0.0
        y_true.append(int(has_fracture))
        y_score.append(score)

    y_true, y_score = np.array(y_true), np.array(y_score)
    auc = roc_auc_score(y_true, y_score)
    fpr, tpr, thresholds = roc_curve(y_true, y_score)
    youden_idx = np.argmax(tpr - fpr)
    best_thr = float(thresholds[youden_idx])
    sensib = float(tpr[youden_idx])
    specif = float(1 - fpr[youden_idx])

    # Bootstrap simple pour l'IC95 de l'AUC (cohérent avec le format des autres modèles)
    rng = np.random.default_rng(seed)
    boot_aucs = []
    n = len(y_true)
    for _ in range(1000):
        idx = rng.integers(0, n, n)
        if len(np.unique(y_true[idx])) < 2:
            continue
        boot_aucs.append(roc_auc_score(y_true[idx], y_score[idx]))
    ci_low, ci_high = np.percentile(boot_aucs, [2.5, 97.5])

    result = {
        'Fracture': {
            'AUC': round(float(auc), 4),
            'IC95': [round(float(ci_low), 4), round(float(ci_high), 4)],
            'Sensibilite': round(sensib, 4),
            'Specificite': round(specif, 4),
            'Seuil_Youden': round(best_thr, 4),
            'mAP50': round(map50, 4),
        },
        'global': {
            'model': 'YOLOv11n',
            'mAP50': round(map50, 4),
            'seed': seed,
            'img_size': IMG_SIZE,
            'preprocessing': f'CLAHE(clip={CLAHE_CLIP})+Letterbox{IMG_SIZE}',
        },
    }
    with open(metrics_path, 'w') as f:
        json.dump(result, f, indent=2)
    results_by_seed[seed] = result
    print(f'✅ Seed {seed} — AUC={auc:.4f} | mAP50={map50:.4f}')

    del model
    gc.collect()
    if torch.cuda.is_available():
        torch.cuda.empty_cache()


✅ Seed 42 : métriques déjà calculées, chargées depuis le disque.
✅ Seed 123 : métriques déjà calculées, chargées depuis le disque.
✅ Seed 2024 : métriques déjà calculées, chargées depuis le disque.
✅ Seed 7 : métriques déjà calculées, chargées depuis le disque.
✅ Seed 77 : métriques déjà calculées, chargées depuis le disque.


## 🏁 Cellule 7 — Agrégation sur les 5 seeds (écart-type ddof = 1)

In [ ]:
aucs = [results_by_seed[s]['Fracture']['AUC'] for s in SEEDS]
maps = [results_by_seed[s]['Fracture']['mAP50'] for s in SEEDS]

agg = {
    'Fracture': {
        'AUC_mean': round(float(np.mean(aucs)), 4),
        'AUC_std': round(float(np.std(aucs, ddof=1)), 4),
        'mAP50_mean': round(float(np.mean(maps)), 4),
        'mAP50_std': round(float(np.std(maps, ddof=1)), 4),
        'n_seeds': len(SEEDS),
    },
    'global': {'model': 'YOLOv11n', 'seeds': SEEDS},
}
with open(os.path.join(SAVE_DIR, 'resultats_agreges_5seeds.json'), 'w') as f:
    json.dump(agg, f, indent=2)

print(f"AUC   = {agg['Fracture']['AUC_mean']:.4f} ± {agg['Fracture']['AUC_std']:.4f}")
print(f"mAP50 = {agg['Fracture']['mAP50_mean']:.4f} ± {agg['Fracture']['mAP50_std']:.4f}")


AUC   = 0.8680 ± 0.0083
mAP50 = 0.3721 ± 0.0281


## 🧪 Cellule — Contrôle : batch effectif (nbs=16 au lieu de 64) — 5 seeds

À exécuter après la cellule d’entraînement des 5 seeds de référence (nbs=64, défaut).

Seul changement : `nbs=16`, égal au batch physique — un ajustement à chaque batch, comme Faster R-CNN/RetinaNet. `batch=16` reste inchangé (nécessaire à la stabilité du BatchNorm interne de YOLO).

5 seeds (42, 123, 2024, 7 , 77), comparés à la référence 5 seeds en écarts-types.

In [10]:
# ═══════════════════════════════════════════════════════════════════════════════
# CONTRÔLE — batch effectif (nbs) — YOLOv11n — 3 seeds
# VERSION CORRIGÉE : vérifie que l'entraînement est réellement allé au bout (via
# results.csv) avant de dire "déjà entraîné", au lieu de se fier à la seule présence
# de best.pt. Reprend proprement depuis last.pt si une session a été coupée.
#
# À COLLER à la place de l'ancienne cellule de contrôle nbs=16 dans le notebook YOLO.
# Réutilise SAVE_DIR, data_yaml_path, LR0, WEIGHT_DECAY, AUG_DEGREES, AUG_TRANSLATE,
# AUG_SCALE, IMG_SIZE, EPOCHS, PATIENCE, SEEDS déjà définis plus haut.
#
# ⚠️ Avant de lancer cette version, supprime (ou renomme) les anciens dossiers
# controle_nbs16/seed_42/ et seed_123/ sur ton Drive : les best.pt qu'ils contiennent
# viennent d'un entraînement interrompu (AUC=0,5 / mAP50=0 constatés) et doivent être
# recalculés, pas réutilisés.
# ═══════════════════════════════════════════════════════════════════════════════
from ultralytics import YOLO
import os, json, numpy as np, pandas as pd

CTRL_SEEDS = [42, 123, 2024,7,77]
CTRL_DIR = os.path.join(SAVE_DIR, 'controle_nbs16')
os.makedirs(CTRL_DIR, exist_ok=True)


def training_is_complete(seed_dir, max_epochs, patience, metric_col='metrics/mAP50(B)'):
    """Un best.pt peut exister après une déconnexion en plein milieu d'une epoch :
    sa seule présence ne prouve rien. On vérifie le nombre réel d'epochs effectuées
    dans results.csv, et si l'arrêt anticipé a bien été déclenché (patience atteinte)."""
    results_csv = os.path.join(seed_dir, 'results.csv')
    if not os.path.exists(results_csv):
        return False
    df = pd.read_csv(results_csv)
    df.columns = df.columns.str.strip()
    n_epochs_done = len(df)
    if n_epochs_done >= max_epochs:
        return True
    if metric_col not in df.columns:
        return False
    best_epoch = df[metric_col].idxmax()
    return (n_epochs_done - 1 - best_epoch) >= patience


ctrl_trained_weights = {}
for seed in CTRL_SEEDS:
    seed_dir = os.path.join(CTRL_DIR, f'seed_{seed}')
    weights_dir = os.path.join(seed_dir, 'weights')
    best_ckpt = os.path.join(weights_dir, 'best.pt')
    last_ckpt = os.path.join(weights_dir, 'last.pt')

    if os.path.exists(best_ckpt) and training_is_complete(seed_dir, EPOCHS, PATIENCE):
        print(f'✅ Seed {seed} (nbs=16) : entraînement réellement complet → {best_ckpt}')
        ctrl_trained_weights[seed] = best_ckpt
        continue

    resume = os.path.exists(last_ckpt)

    def _train(resume_flag, model_src):
        m = YOLO(model_src)
        m.train(
            data=data_yaml_path,
            imgsz=IMG_SIZE,
            epochs=EPOCHS,
            patience=PATIENCE,
            batch=16,
            nbs=16,                 # ← seul changement vs la référence (défaut ultralytics = 64)
            seed=seed,
            project=CTRL_DIR,
            name=f'seed_{seed}',
            exist_ok=True,
            resume=resume_flag,
            optimizer='SGD',
            lr0=LR0,
            momentum=0.9,
            weight_decay=WEIGHT_DECAY,
            hsv_h=0.0, hsv_s=0.0, hsv_v=0.0,
            degrees=AUG_DEGREES, translate=AUG_TRANSLATE, scale=AUG_SCALE,
            shear=0.0, perspective=0.0,
            flipud=0.0, fliplr=0.5,
            mosaic=0.0, mixup=0.0,
        )

    if resume:
        print(f'\n🔄 Seed {seed} — entraînement incomplet détecté (results.csv), '
              f'reprise depuis last.pt (contrôle nbs=16)...')
        try:
            _train(True, last_ckpt)
        except AssertionError as e:
            # last.pt dit en interne "déjà fini" alors que results.csv disait le contraire
            # (fichiers désynchronisés par une déconnexion). On repart proprement de zéro.
            print(f'⚠️ {e}')
            print(f'   → last.pt indique un entraînement déjà terminé malgré results.csv. '
                  f'Nouvel entraînement complet depuis les poids COCO.')
            _train(False, 'yolo11n.pt')
    else:
        print(f'\n🚀 Seed {seed} — entraînement YOLOv11n (contrôle nbs=16)...')
        _train(False, 'yolo11n.pt')

    ctrl_trained_weights[seed] = best_ckpt
    print(f'✅ Seed {seed} — terminé → {best_ckpt}')


# ── Évaluation (même logique que la cellule d'évaluation de référence du notebook) ──
def evaluate_yolo_checkpoint(weights_path):
    model = YOLO(weights_path)
    val_results = model.val(data=data_yaml_path, split='test', imgsz=IMG_SIZE, iou=0.5, conf=0.001)
    map50 = float(val_results.box.map50)

    test_dir = os.path.join(YOLO_DATA, 'images', 'test')
    labels_dir = os.path.join(YOLO_DATA, 'labels', 'test')
    y_true, y_score = [], []
    for fname in sorted(os.listdir(test_dir)):
        stem = os.path.splitext(fname)[0]
        lbl_path = os.path.join(labels_dir, stem + '.txt')
        has_fracture = int(os.path.exists(lbl_path) and os.path.getsize(lbl_path) > 0)
        y_true.append(has_fracture)
        pred = model.predict(os.path.join(test_dir, fname), imgsz=IMG_SIZE, conf=0.001, verbose=False)[0]
        score = float(pred.boxes.conf.max()) if len(pred.boxes) > 0 else 0.0
        y_score.append(score)

    from sklearn.metrics import roc_auc_score, roc_curve
    y_true, y_score = np.array(y_true), np.array(y_score)
    auc = roc_auc_score(y_true, y_score)
    fpr, tpr, thr = roc_curve(y_true, y_score)
    j = int(np.argmax(tpr - fpr))
    return {'AUC': round(float(auc), 4), 'mAP50': round(map50, 4),
            'Sensibilite': round(float(tpr[j]), 4), 'Specificite': round(float(1 - fpr[j]), 4)}


# Ré-évalue aussi si les métriques sauvegardées correspondent à un ancien checkpoint cassé
ctrl_results = {}
for seed in CTRL_SEEDS:
    mpath = os.path.join(CTRL_DIR, f'metriques_seed{seed}.json')
    if os.path.exists(mpath):
        with open(mpath) as f:
            cached = json.load(f)
        if cached['AUC'] == 0.5 and cached['mAP50'] == 0.0:
            print(f'⚠️ Seed {seed} : métriques en cache suspectes (AUC=0,5, mAP50=0) → ré-évaluation forcée.')
        else:
            ctrl_results[seed] = cached
            continue
    print(f'📊 Seed {seed} (nbs=16) — évaluation...')
    ctrl_results[seed] = evaluate_yolo_checkpoint(ctrl_trained_weights[seed])
    with open(mpath, 'w') as f:
        json.dump(ctrl_results[seed], f, indent=2)

# ── Comparaison à la référence 5 seeds (nbs=64, déjà entraînée plus haut) ──
ref = {}
for seed in SEEDS:
    p = os.path.join(SAVE_DIR, f'metriques_seed{seed}.json')
    if os.path.exists(p):
        with open(p) as f:
            m = json.load(f)['Fracture']
        ref[seed] = (m['AUC'], m['mAP50'])

aucs = np.array([v[0] for v in ref.values()])
maps = np.array([v[1] for v in ref.values()])
ctrl_aucs = np.array([r['AUC'] for r in ctrl_results.values()])
ctrl_maps = np.array([r['mAP50'] for r in ctrl_results.values()])

print('\n' + '=' * 78)
print('RÉFÉRENCE YOLOv11n (nbs=64, défaut) — 5 seeds')
for s_, (a, m) in ref.items():
    print(f'  seed {s_:>5} : AUC={a:.4f} | mAP50={m:.4f}')
print(f'  moyenne ± écart-type : AUC={aucs.mean():.4f} ± {aucs.std():.4f} | mAP50={maps.mean():.4f} ± {maps.std():.4f}')

print('\nCONTRÔLE nbs=16 — 3 seeds')
for seed, r in ctrl_results.items():
    print(f"  seed {seed:>5} : AUC={r['AUC']:.4f} | mAP50={r['mAP50']:.4f}")
print(f'  moyenne : AUC={ctrl_aucs.mean():.4f} | mAP50={ctrl_maps.mean():.4f}')

d_auc = (ctrl_aucs.mean() - aucs.mean()) / max(aucs.std(), 1e-9)
d_map = (ctrl_maps.mean() - maps.mean()) / max(maps.std(), 1e-9)
print(f'\nÉcart moyen du contrôle vs référence : ΔAUC = {d_auc:+.2f} σ | ΔmAP50 = {d_map:+.2f} σ')
print('Lecture : |Δ| < 1 σ → effet non détectable (documenter, pas de réentraînement complet).')
print('          1-2 σ → ambigu, ajouter des seeds.  |Δ| > 2 σ → effet réel probable (réentraîner en 5 seeds).')
print('=' * 78)

✅ Seed 42 (nbs=16) : entraînement réellement complet → /content/drive/MyDrive/resultats_finaux_fracture/YOLOv11n/controle_nbs16/seed_42/weights/best.pt
✅ Seed 123 (nbs=16) : entraînement réellement complet → /content/drive/MyDrive/resultats_finaux_fracture/YOLOv11n/controle_nbs16/seed_123/weights/best.pt
✅ Seed 2024 (nbs=16) : entraînement réellement complet → /content/drive/MyDrive/resultats_finaux_fracture/YOLOv11n/controle_nbs16/seed_2024/weights/best.pt

🔄 Seed 7 — entraînement incomplet détecté (results.csv), reprise depuis last.pt (contrôle nbs=16)...
New https://pypi.org/project/ultralytics/8.4.174 available 😃 Update with 'pip install -U ultralytics'
Ultralytics 8.3.40 🚀 Python-3.13.15 torch-2.11.0+cu130 CUDA:0 (Tesla T4, 14913MiB)
engine/trainer: task=detect, mode=train, model=/content/drive/MyDrive/resultats_finaux_fracture/YOLOv11n/controle_nbs16/seed_7/weights/last.pt, data=/content/yolo_fracatlas_dataset/data.yaml, epochs=150, time=None, patience=20, batch=16, imgsz=640, sa

100%|██████████| 755k/755k [00:00<00:00, 20.4MB/s]


TensorBoard: Start with 'tensorboard --logdir /content/drive/MyDrive/resultats_finaux_fracture/YOLOv11n/controle_nbs16/seed_7', view at http://localhost:6006/

                   from  n    params  module                                       arguments                     
  0                  -1  1       464  ultralytics.nn.modules.conv.Conv             [3, 16, 3, 2]                 
  1                  -1  1      4672  ultralytics.nn.modules.conv.Conv             [16, 32, 3, 2]                
  2                  -1  1      6640  ultralytics.nn.modules.block.C3k2            [32, 64, 1, False, 0.25]      
  3                  -1  1     36992  ultralytics.nn.modules.conv.Conv             [64, 64, 3, 2]                
  4                  -1  1     26080  ultralytics.nn.modules.block.C3k2            [64, 128, 1, False, 0.25]     
  5                  -1  1    147712  ultralytics.nn.modules.conv.Conv             [128, 128, 3, 2]              
  6                  -1  1     87040  ultr

100%|██████████| 5.35M/5.35M [00:00<00:00, 99.1MB/s]


AMP: checks passed ✅


train: Scanning /content/yolo_fracatlas_dataset/labels/train... 3883 images, 2344 backgrounds, 0 corrupt: 100%|██████████| 3883/3883 [00:01<00:00, 2836.82it/s]


... (276 outputs left out)


                 Class     Images  Instances      Box(P          R      mAP50  mAP50-95): 100%|██████████| 39/39 [00:06<00:00,  6.48it/s]


                   all        613        134      0.635      0.351      0.374      0.143
Speed: 0.3ms preprocess, 4.9ms inference, 0.0ms loss, 2.0ms postprocess per image
Results saved to runs/detect/val
📊 Seed 77 (nbs=16) — évaluation...
Ultralytics 8.3.40 🚀 Python-3.13.15 torch-2.11.0+cu130 CUDA:0 (Tesla T4, 14913MiB)
YOLO11n summary (fused): 238 layers, 2,582,347 parameters, 0 gradients, 6.3 GFLOPs


val: Scanning /content/yolo_fracatlas_dataset/labels/test.cache... 613 images, 505 backgrounds, 0 corrupt: 100%|██████████| 613/613 [00:00<?, ?it/s]
                 Class     Images  Instances      Box(P          R      mAP50  mAP50-95): 100%|██████████| 39/39 [00:05<00:00,  7.15it/s]


                   all        613        134      0.684       0.42      0.444      0.169
Speed: 0.3ms preprocess, 4.4ms inference, 0.0ms loss, 1.6ms postprocess per image
Results saved to runs/detect/val2

RÉFÉRENCE YOLOv11n (nbs=64, défaut) — 5 seeds
  seed    42 : AUC=0.8640 | mAP50=0.3434
  seed   123 : AUC=0.8642 | mAP50=0.3848
  seed  2024 : AUC=0.8776 | mAP50=0.3621
  seed     7 : AUC=0.8775 | mAP50=0.3494
  seed    77 : AUC=0.8567 | mAP50=0.4206
  moyenne ± écart-type : AUC=0.8680 ± 0.0083 | mAP50=0.3721 ± 0.0281

CONTRÔLE nbs=16 — 3 seeds
  seed    42 : AUC=0.8828 | mAP50=0.4133
  seed   123 : AUC=0.8912 | mAP50=0.4206
  seed  2024 : AUC=0.8840 | mAP50=0.4035
  seed     7 : AUC=0.8785 | mAP50=0.3743
  seed    77 : AUC=0.8512 | mAP50=0.4442
  moyenne : AUC=0.8775 | mAP50=0.4112

Écart moyen du contrôle vs référence : ΔAUC = +1.16 σ | ΔmAP50 = +1.39 σ
Lecture : |Δ| < 1 σ → effet non détectable (documenter, pas de réentraînement complet).
          1-2 σ → ambigu, ajouter des seed

## 🔬 Cellule — Analyse des erreurs (YOLOv11n, configuration retenue nbs=64, 5 seeds)

À exécuter après la Cellule 5 (poids de référence dans `seed_X/weights/best.pt`, nbs=64 par défaut). **Vérifie d'abord `args.yaml` et `results.csv` de chaque graine** (réglages conformes au tableau 1, entraînement terminé) et s'arrête sinon. Classe chaque fracture du test (détectée / score trop bas / mal localisée / non détectée) et chaque faux positif, avec ventilation par région, taille, matériel et fractures multiples. Seuil de confiance de chaque seed choisi sur la validation (F1 maximal). Résultats écrits dans `resultats_finaux_fracture/Erreurs/` avec le suffixe `_nbs64` (les fichiers nbs=16 ne sont pas écrasés). Écarts-types avec ddof = 1.

In [ ]:
# ══════════════════════════════════════════════════════════════════════════════
# ANALYSE DES ERREURS — YOLOv11n, configuration retenue (nbs=64), 5 seeds
# Réutilise : SAVE_DIR, YOLO_DATA, IMG_SIZE, SEEDS, FRACATLAS_ROOT (déjà définis plus haut)
# Poids : <SAVE_DIR>/seed_X/weights/best.pt  (entraînement de référence, nbs=64 = défaut Ultralytics)
# Résultats : <resultats_finaux_fracture>/Erreurs/  (CSV + synthèse + images d'exemples)
# ══════════════════════════════════════════════════════════════════════════════
import os, glob, json, pickle, gc, yaml
import numpy as np, pandas as pd, cv2, torch
from tqdm.auto import tqdm
from ultralytics import YOLO
from sklearn.metrics import roc_auc_score

MODEL_NAME = 'YOLOv11n'
W_DIR   = SAVE_DIR            # entraînement de référence (nbs=64) — PAS controle_nbs16
ERR_DIR = os.path.join(os.path.dirname(SAVE_DIR), 'Erreurs'); os.makedirs(ERR_DIR, exist_ok=True)
IOU_THR, MIN_CONF, FIXED_CONF = 0.5, 0.001, 0.25      # IoU = celui du mAP@50 ; seuil fixe si pas de validation
# predict() : NMS par défaut d'Ultralytics (iou=0.7), 300 boîtes max, score >= 0.001 (section 2.5)
weights = {s: os.path.join(W_DIR, f'seed_{s}', 'weights', 'best.pt') for s in SEEDS}
for s, p in weights.items():
    assert os.path.exists(p), f'Poids introuvables (seed {s}) : {p}'

RUN_SEEDS, MODEL_DIR = list(SEEDS), W_DIR
# ── Vérification avant de lancer : hyperparamètres réellement utilisés + entraînement terminé ──
EXPECTED_NBS = 64
EXPECTED_ARGS = dict(optimizer='SGD', lr0=LR0, momentum=0.9, weight_decay=WEIGHT_DECAY, batch=16, nbs=EXPECTED_NBS,
                     epochs=EPOCHS, patience=PATIENCE, imgsz=IMG_SIZE, mosaic=0.0, mixup=0.0,
                     degrees=AUG_DEGREES, translate=AUG_TRANSLATE, scale=AUG_SCALE, shear=0.0, perspective=0.0,
                     flipud=0.0, fliplr=0.5, hsv_h=0.0, hsv_s=0.0, hsv_v=0.0)

def _check_args(a, expected):
    bad = []
    for k, v in expected.items():
        got = a.get(k)
        if isinstance(v, (int, float)) and got is not None and not isinstance(got, str):
            same = abs(float(got) - float(v)) <= 1e-9 * max(1.0, abs(float(v)))
        else:
            same = (got == v)
        if not same: bad.append(f'{k}: args.yaml = {got!r} ≠ attendu {v!r}')
    return bad

def _complete(seed_dir):
    """Entraînement réellement terminé ? (results.csv : 150 époques, ou arrêt anticipé par la patience)"""
    p = os.path.join(seed_dir, 'results.csv')
    if not os.path.exists(p): return False, 0, -1
    df = pd.read_csv(p); df.columns = df.columns.str.strip()
    n = len(df)
    if 'metrics/mAP50(B)' in df.columns and 'metrics/mAP50-95(B)' in df.columns:
        fit = 0.1 * df['metrics/mAP50(B)'] + 0.9 * df['metrics/mAP50-95(B)']      # fitness ultralytics
        best = int(fit.idxmax())
    else:
        best = -1
    return (n >= EPOCHS or (best >= 0 and (n - 1 - best) >= PATIENCE)), n, best

print(f"{'seed':<6}{'nbs':<6}{'batch':<7}{'optim.':<8}{'lr0':<9}{'époques':<9}{'meilleure':<10}statut")
_args = {}
for _s in RUN_SEEDS:
    _d = os.path.join(MODEL_DIR, f'seed_{_s}')
    _p = os.path.join(_d, 'args.yaml')
    assert os.path.exists(_p), f'args.yaml introuvable pour la graine {_s} : {_p}'
    _a = yaml.safe_load(open(_p)); _args[_s] = _a
    _ok, _n, _b = _complete(_d)
    print(f"{_s:<6}{str(_a.get('nbs')):<6}{str(_a.get('batch')):<7}{str(_a.get('optimizer')):<8}{str(_a.get('lr0')):<9}"
          f"{_n:<9}{_b + 1:<10}{'terminé' if _ok else 'INCOMPLET'}")
    _bad = _check_args(_a, EXPECTED_ARGS)
    assert not _bad, f'Graine {_s} : réglages différents de ceux du tableau 1 →\n  ' + '\n  '.join(_bad)
    assert _ok, f'Graine {_s} : entraînement incomplet ({_n} époques) : relancer la Cellule 5 pour cette graine.'
print(f'✅ nbs={EXPECTED_NBS}, SGD, lr0={LR0}, batch=16, entraînements terminés : conformes au tableau 1 pour les {len(RUN_SEEDS)} graines.\n')

# ── Jeux de test et de validation déjà prétraités (CLAHE + letterbox) ──
def load_split(split):
    img_dir, lbl_dir = os.path.join(YOLO_DATA, 'images', split), os.path.join(YOLO_DATA, 'labels', split)
    paths = {os.path.splitext(f)[0]: os.path.join(img_dir, f) for f in sorted(os.listdir(img_dir))}
    gts = {}
    for s in paths:
        lp = os.path.join(lbl_dir, s + '.txt')
        a = np.loadtxt(lp, ndmin=2) if os.path.exists(lp) and os.path.getsize(lp) > 0 else np.zeros((0, 5))
        xc, yc, w, h = (a[:, i] * IMG_SIZE for i in (1, 2, 3, 4))
        gts[s] = np.c_[xc - w / 2, yc - h / 2, xc + w / 2, yc + h / 2] if len(a) else np.zeros((0, 4))
    return paths, gts

TEST_P, TEST_GT = load_split('test'); VAL_P, VAL_GT = load_split('val')
n_gt = sum(len(g) for g in TEST_GT.values())
print(f"TEST : {len(TEST_P)} images dont {sum(len(g) > 0 for g in TEST_GT.values())} fracturées ({n_gt} fractures annotées)  [attendu : 613 dont 108]")

# ── Appariement (IoU >= 0,5, glouton par score décroissant) ──
def iou_matrix(a, b):
    if len(a) == 0 or len(b) == 0: return np.zeros((len(a), len(b)))
    x1 = np.maximum(a[:, None, 0], b[None, :, 0]); y1 = np.maximum(a[:, None, 1], b[None, :, 1])
    x2 = np.minimum(a[:, None, 2], b[None, :, 2]); y2 = np.minimum(a[:, None, 3], b[None, :, 3])
    inter = np.clip(x2 - x1, 0, None) * np.clip(y2 - y1, 0, None)
    aa = (a[:, 2] - a[:, 0]) * (a[:, 3] - a[:, 1]); ab = (b[:, 2] - b[:, 0]) * (b[:, 3] - b[:, 1])
    return inter / (aa[:, None] + ab[None, :] - inter + 1e-9)

def match(gt, boxes, scores, thr):
    k = scores >= thr; pb, ps = boxes[k], scores[k]
    o = np.argsort(-ps); pb, ps = pb[o], ps[o]
    tp = np.zeros(len(pb), bool); ok = np.zeros(len(gt), bool)
    if len(gt) and len(pb):
        iou = iou_matrix(pb, gt)
        for i in range(len(pb)):
            free = np.where(~ok)[0]
            if len(free) == 0: break
            j = free[np.argmax(iou[i, free])]
            if iou[i, j] >= IOU_THR: ok[j] = True; tp[i] = True
    return pb, ps, tp, ok

def best_threshold(preds, gts):       # seuil de confiance = F1 maximal sur la validation
    best, bt = -1, FIXED_CONF
    for t in np.arange(0.05, 0.96, 0.05):
        TP = FP = FN = 0
        for s, (b, sc) in preds.items():
            _, _, tp, ok = match(gts[s], b, sc, t); TP += tp.sum(); FP += (~tp).sum(); FN += (~ok).sum()
        f1 = 2 * TP / (2 * TP + FP + FN + 1e-9)
        if f1 > best: best, bt = f1, float(t)
    return bt

# ── Prédictions (sauvegardées sur Drive : reprise possible après une coupure) ──
cache_path = os.path.join(ERR_DIR, 'preds_yolov11n_nbs64.pkl')
cache = pickle.load(open(cache_path, 'rb')) if os.path.exists(cache_path) else {'test': {}, 'thr': {}}
def predict_all(model, paths, desc):
    out = {}
    for s, p in tqdm(paths.items(), desc=desc, leave=False):
        r = model.predict(p, imgsz=IMG_SIZE, conf=MIN_CONF, verbose=False)[0]
        out[s] = (r.boxes.xyxy.cpu().numpy(), r.boxes.conf.cpu().numpy())
    return out
for seed in SEEDS:
    if seed in cache['thr']: print(f'✅ seed {seed} : prédictions déjà faites'); continue
    print(f'🔎 seed {seed} — prédictions test + validation...')
    model = YOLO(weights[seed])
    cache['test'][seed] = predict_all(model, TEST_P, 'test')
    cache['thr'][seed] = best_threshold(predict_all(model, VAL_P, 'val'), VAL_GT)
    pickle.dump(cache, open(cache_path, 'wb'))
    print(f"   seuil de confiance retenu (F1 max sur la validation) : {cache['thr'][seed]:.2f}")
    del model; gc.collect(); torch.cuda.empty_cache()

# ── Contrôle : l'AUC recalculée ici doit retrouver celle de la cellule d'évaluation ──
for seed in SEEDS:
    mp = os.path.join(W_DIR, f'metriques_seed{seed}.json')
    if os.path.exists(mp):
        y = [int(len(TEST_GT[s]) > 0) for s in TEST_P]
        sc = [float(cache['test'][seed][s][1].max()) if len(cache['test'][seed][s][1]) else 0.0 for s in TEST_P]
        a_ici, a_ref = roc_auc_score(y, sc), json.load(open(mp))['AUC']
        print(f'   contrôle seed {seed} : AUC ici = {a_ici:.4f} | AUC évaluation = {a_ref:.4f}', '✅' if abs(a_ici - a_ref) < 2e-3 else '⚠️ ÉCART')

# ── Métadonnées FracAtlas : région, matériel, fractures multiples ──
gt_df = pd.DataFrame([dict(stem=s, gt_idx=j, rel_area=(b[2] - b[0]) * (b[3] - b[1]) / IMG_SIZE ** 2)
                      for s, g in TEST_GT.items() for j, b in enumerate(g)])
gt_df['taille'] = pd.qcut(gt_df.rel_area.rank(method='first'), 3, labels=['petite', 'moyenne', 'grande']).astype(str)
for c in ('région', 'matériel', 'fractures multiples'): gt_df[c] = 'inconnu'
mc = glob.glob(f'{FRACATLAS_ROOT}/**/dataset.csv', recursive=True)
if mc:
    meta = pd.read_csv(mc[0]); meta['stem'] = meta['image_id'].astype(str).map(lambda x: os.path.splitext(x)[0])
    def _reg(r):
        if r.get('mixed', 0) == 1: return 'mixte'
        for c, lab in (('hand', 'main'), ('shoulder', 'épaule'), ('leg', 'jambe'), ('hip', 'hanche')):
            if r.get(c, 0) == 1: return lab
        return 'inconnu'
    meta['région'] = meta.apply(_reg, axis=1)
    meta['matériel'] = np.where(meta.get('hardware', 0) == 1, 'avec matériel', 'sans matériel')
    meta['fractures multiples'] = np.where(meta.get('fracture_count', 1) > 1, 'plusieurs', 'une seule')
    gt_df = gt_df.drop(columns=['région', 'matériel', 'fractures multiples']).merge(
        meta[['stem', 'région', 'matériel', 'fractures multiples']], on='stem', how='left').fillna('inconnu')
else:
    print('⚠️ dataset.csv introuvable : pas de ventilation par région / matériel')

# ── Classement des erreurs ──
gt_rows, fp_rows, runs = [], [], []
for seed in SEEDS:
    thr, TP, FPn = cache['thr'][seed], 0, 0
    for s, (b, sc) in cache['test'][seed].items():
        g = TEST_GT[s]; pb, ps, tp, ok = match(g, b, sc, thr); TP += int(tp.sum()); FPn += int((~tp).sum())
        iou_all, iou_kept = iou_matrix(b, g), iou_matrix(pb, g)
        for j in range(len(g)):
            if ok[j]: status = 'détectée'
            elif len(b) and (iou_all[:, j][sc < thr] >= IOU_THR).any(): status = 'score trop bas'
            elif len(pb) and iou_kept[:, j].max() >= 0.1: status = 'mal localisée'
            else: status = 'non détectée'
            gt_rows.append(dict(model=MODEL_NAME, seed=seed, stem=s, gt_idx=j, detected=bool(ok[j]), status=status,
                                best_iou=float(iou_all[:, j].max()) if len(b) else 0.0))
        for i in np.where(~tp)[0]:
            if len(g) == 0: kind = 'image sans fracture'
            else:
                mx = iou_kept[i].max()
                kind = 'doublon' if mx >= IOU_THR else ('localisation imprécise' if mx >= 0.1 else 'autre région')
            fp_rows.append(dict(model=MODEL_NAME, seed=seed, stem=s, score=float(ps[i]), kind=kind))
    FN = n_gt - TP; P, R = TP / (TP + FPn + 1e-9), TP / (TP + FN + 1e-9)
    runs.append(dict(model=MODEL_NAME, seed=seed, seuil=thr, TP=TP, FP=FPn, FN=FN, precision=P, rappel=R, F1=2 * P * R / (P + R + 1e-9)))

gt_res = pd.DataFrame(gt_rows).merge(gt_df, on=['stem', 'gt_idx']); fp_res = pd.DataFrame(fp_rows); runs = pd.DataFrame(runs)
tag = 'yolov11n_nbs64'
gt_res.to_csv(f'{ERR_DIR}/detail_fractures_{tag}.csv', index=False); fp_res.to_csv(f'{ERR_DIR}/detail_faux_positifs_{tag}.csv', index=False)
runs.to_csv(f'{ERR_DIR}/bilan_par_graine_{tag}.csv', index=False); gt_df.to_csv(f'{ERR_DIR}/fractures_annotees.csv', index=False)

# ── Tableaux de synthèse (écart-type avec correction de Bessel, ddof=1, comme dans l'article) ──
ms = lambda s: f'{s.mean():.3f} ± {s.std(ddof=1):.3f}'
nS = len(SEEDS)
out = [('Tableau A. Bilan par graine', runs.round(3).set_index('seed')),
       ('Tableau A bis. Bilan moyen (moyenne ± écart-type)', pd.DataFrame({
           'seuil': [runs.seuil.mean()], 'TP': [runs.TP.mean()], 'FP': [runs.FP.mean()], 'FN': [runs.FN.mean()],
           'précision': [ms(runs.precision)], 'rappel': [ms(runs.rappel)], 'F1': [ms(runs.F1)]}, index=[MODEL_NAME]).round(2)),
       ('Tableau B. Statut des fractures annotées (nombre moyen par graine)',
        (gt_res.groupby('status').size() / nS).round(1).to_frame(MODEL_NAME))]
for fac in ['région', 'taille', 'matériel', 'fractures multiples']:
    r = ((1 - gt_res.groupby([fac, 'seed']).detected.mean().groupby(fac).mean()) * 100).round(1).to_frame('FN (%)')
    r.insert(0, 'n fractures', gt_df.groupby(fac).size()); out.append((f'Tableau C. Taux de faux négatifs selon : {fac}', r))
if len(fp_res):
    out.append(('Tableau D. Nature des faux positifs (nombre moyen par graine)', (fp_res.groupby('kind').size() / nS).round(1).to_frame(MODEL_NAME)))
never = gt_res.groupby(['stem', 'gt_idx']).detected.sum(); never = int((never == 0).sum())
with open(f'{ERR_DIR}/synthese_{tag}.md', 'w', encoding='utf-8') as f:
    for titre, t in out:
        f.write(f'### {titre}\n\n{t.to_markdown() if hasattr(t, "to_markdown") else t.to_string()}\n\n'); print(f'\n{titre}\n{t}')
    f.write(f'Fractures jamais détectées (aucune graine) : {never}\n')
print(f'\nFractures jamais détectées par aucune des {nS} graines : {never} / {len(gt_df)}')
print('Contrôle TP + FN = fractures annotées :', bool(((runs.TP + runs.FN) == n_gt).all()), f'({n_gt})')

# ── Images d'exemples (vert = annotation, rouge = détection) ──
ex = f'{ERR_DIR}/exemples_{tag}'; os.makedirs(ex, exist_ok=True)
s0 = SEEDS[0]; thr0 = cache['thr'][s0]
nv = gt_res.groupby(['stem', 'gt_idx']).detected.sum(); nv = list(nv[nv == 0].index.get_level_values(0).unique()[:5])
tf = list(fp_res[fp_res.seed == s0].sort_values('score', ascending=False).drop_duplicates('stem').stem[:3]) if len(fp_res) else []
for s in nv + tf:
    img = cv2.imread(TEST_P[s])
    for b in TEST_GT[s]: cv2.rectangle(img, tuple(map(int, b[:2])), tuple(map(int, b[2:])), (0, 200, 0), 2)
    for bb, c in zip(*cache['test'][s0][s]):
        if c >= thr0:
            cv2.rectangle(img, tuple(map(int, bb[:2])), tuple(map(int, bb[2:])), (0, 0, 255), 2)
            cv2.putText(img, f'{c:.2f}', (int(bb[0]), max(int(bb[1]) - 4, 10)), 0, 0.5, (0, 0, 255), 1)
    cv2.imwrite(f'{ex}/{s}.png', img)
print(f'\n✅ Terminé. Résultats dans {ERR_DIR} → envoie-moi synthese_{tag}.md, bilan_par_graine_{tag}.csv et detail_fractures_{tag}.csv')


## 📐 Cellule 8 — Précision, rappel et F1 (boîtes) — YOLOv11n, nbs=64, 5 graines

À lancer après la Cellule 6 (réutilise `YOLO_DATA`, `SAVE_DIR`, `data_yaml_path`, `IMG_SIZE`, `SEEDS`). Poids : `SAVE_DIR/seed_X/weights/best.pt` (entraînement de référence).

- **Vérification préalable** : lit `args.yaml` et `results.csv` de chaque graine ; s'arrête si les réglages diffèrent du tableau 1 (SGD, lr0, batch 16, nbs 64…) ou si l'entraînement est incomplet.
- `predict()` utilise le NMS par défaut d'Ultralytics (0,7), comme la Cellule 6 et l'analyse des erreurs : les TP/FP/FN doivent retrouver le tableau A bis de l'analyse des erreurs.
- Une prédiction est un vrai positif si son IoU avec une boîte annotée est ≥ 0,5 (appariement glouton par confiance décroissante, une boîte annotée ne peut être appariée qu'une fois).
- Le seuil de confiance maximise le **F1 de validation** (boîtes), sur la grille 0,05–0,95 par pas de 0,05 (identique à la section 2.5), puis il est appliqué au test.
- Les images normales comptent : toute boîte prédite dessus est un faux positif. On rapporte aussi les FP par image normale et la spécificité image à ce seuil.
- Le mAP@50 est recalculé et comparé à celui de `metriques_seed{seed}.json`.
- Moyenne ± écart-type avec **ddof = 1** (n = 5 graines). Sauvegarde dans `metriques_detection_seed{seed}.json` et `resultats_detection_agreges.json`.

In [ ]:
# ═══════════════════════════════════════════════════════════════════════════
# Cellule 8 — Précision / Rappel / F1 au niveau des boîtes (YOLOv11n, 5 graines)
# Seuil de confiance choisi sur la VALIDATION (maximise le F1), appliqué au TEST.
# Poids utilisés : entraînement de référence (SAVE_DIR/seed_X/weights/best.pt, nbs=64 = défaut Ultralytics)
# ═══════════════════════════════════════════════════════════════════════════
import os, json, gc
import numpy as np
import pandas as pd
import torch
from ultralytics import YOLO
import yaml

MODEL_DIR = SAVE_DIR            # entraînement de référence (seed_X/weights/best.pt)
RUN_SEEDS = list(SEEDS)         # mêmes 5 graines que la Cellule 1
EXPECTED_NBS = 64               # défaut Ultralytics (batch=16 → 4 accumulations)
IOU_MATCH = 0.5      # une prédiction est un vrai positif si IoU >= 0.5 avec une boîte annotée
# predict() : NMS par défaut d'Ultralytics (iou=0.7), comme la Cellule 6 (AUC) et l'analyse des erreurs
CONF_MIN  = 0.001    # on garde tout, le seuil est appliqué ensuite
THR_GRID  = np.round(np.arange(0.05, 0.951, 0.05), 2)   # même grille que la section 2.5 de l'article (0,05 à 0,95, pas de 0,05)
# égalité de F1 → np.argmax retient le seuil le plus faible, comme décrit en 2.5


def _load_gt(lbl_path, size):
    """Boîtes annotées (xyxy, pixels de l'image letterboxée). Vide si image normale."""
    if not os.path.exists(lbl_path) or os.path.getsize(lbl_path) == 0:
        return np.zeros((0, 4))
    a = np.loadtxt(lbl_path, ndmin=2)
    xc, yc, w, h = a[:, 1] * size, a[:, 2] * size, a[:, 3] * size, a[:, 4] * size
    return np.stack([xc - w / 2, yc - h / 2, xc + w / 2, yc + h / 2], axis=1)


def _iou_matrix(a, b):
    if len(a) == 0 or len(b) == 0:
        return np.zeros((len(a), len(b)))
    x1 = np.maximum(a[:, None, 0], b[None, :, 0]); y1 = np.maximum(a[:, None, 1], b[None, :, 1])
    x2 = np.minimum(a[:, None, 2], b[None, :, 2]); y2 = np.minimum(a[:, None, 3], b[None, :, 3])
    inter = np.clip(x2 - x1, 0, None) * np.clip(y2 - y1, 0, None)
    area_a = (a[:, 2] - a[:, 0]) * (a[:, 3] - a[:, 1])
    area_b = (b[:, 2] - b[:, 0]) * (b[:, 3] - b[:, 1])
    return inter / (area_a[:, None] + area_b[None, :] - inter + 1e-9)


CHUNK = 16   # images par lot : évite d'envoyer tout le split d'un coup sur le GPU (OOM)


def _collect(model, split):
    """Prédictions (toutes confiances >= CONF_MIN) + vérité terrain, image par image."""
    img_dir = os.path.join(YOLO_DATA, 'images', split)
    lbl_dir = os.path.join(YOLO_DATA, 'labels', split)
    fnames = sorted(os.listdir(img_dir))
    data = []
    for i in range(0, len(fnames), CHUNK):
        chunk = fnames[i:i + CHUNK]
        results = model.predict([os.path.join(img_dir, f) for f in chunk], imgsz=IMG_SIZE,
                                conf=CONF_MIN, batch=CHUNK, verbose=False)
        for fname, r in zip(chunk, results):
            boxes = r.boxes.xyxy.cpu().numpy() if len(r.boxes) else np.zeros((0, 4))
            scores = r.boxes.conf.cpu().numpy() if len(r.boxes) else np.zeros(0)
            order = np.argsort(-scores)
            boxes, scores = boxes[order], scores[order]
            gt = _load_gt(os.path.join(lbl_dir, os.path.splitext(fname)[0] + '.txt'), IMG_SIZE)
            data.append({'scores': scores, 'n_gt': len(gt), 'iou': _iou_matrix(boxes, gt)})
        del results
    return data


def _counts(data, thr):
    """TP/FP/FN au niveau des boîtes, appariement glouton par confiance décroissante."""
    tp = fp = fn = 0
    n_normal = fp_normal = normal_flagged = 0
    for d in data:
        n_pred = int((d['scores'] >= thr).sum())     # scores triés : les n_pred premiers
        n_gt = d['n_gt']
        taken = np.zeros(n_gt, dtype=bool)
        tp_i = 0
        for i in range(n_pred):
            if n_gt == 0:
                break
            ious = np.where(taken, -1.0, d['iou'][i])
            j = int(np.argmax(ious))
            if ious[j] >= IOU_MATCH:
                taken[j] = True
                tp_i += 1
        tp += tp_i; fp += n_pred - tp_i; fn += n_gt - tp_i
        if n_gt == 0:
            n_normal += 1; fp_normal += n_pred; normal_flagged += int(n_pred > 0)
    prec = tp / (tp + fp) if (tp + fp) else 0.0
    rec = tp / (tp + fn) if (tp + fn) else 0.0
    f1 = 2 * prec * rec / (prec + rec) if (prec + rec) else 0.0
    return {'TP': tp, 'FP': fp, 'FN': fn, 'Precision': prec, 'Rappel': rec, 'F1': f1,
            'FP_par_image_normale': fp_normal / n_normal if n_normal else float('nan'),
            'Specificite_image': 1 - normal_flagged / n_normal if n_normal else float('nan')}


gc.collect()
if torch.cuda.is_available():
    torch.cuda.empty_cache()

# ── Vérification avant de lancer : hyperparamètres réellement utilisés + entraînement terminé ──
EXPECTED_ARGS = dict(optimizer='SGD', lr0=LR0, momentum=0.9, weight_decay=WEIGHT_DECAY, batch=16, nbs=EXPECTED_NBS,
                     epochs=EPOCHS, patience=PATIENCE, imgsz=IMG_SIZE, mosaic=0.0, mixup=0.0,
                     degrees=AUG_DEGREES, translate=AUG_TRANSLATE, scale=AUG_SCALE, shear=0.0, perspective=0.0,
                     flipud=0.0, fliplr=0.5, hsv_h=0.0, hsv_s=0.0, hsv_v=0.0)

def _check_args(a, expected):
    bad = []
    for k, v in expected.items():
        got = a.get(k)
        if isinstance(v, (int, float)) and got is not None and not isinstance(got, str):
            same = abs(float(got) - float(v)) <= 1e-9 * max(1.0, abs(float(v)))
        else:
            same = (got == v)
        if not same: bad.append(f'{k}: args.yaml = {got!r} ≠ attendu {v!r}')
    return bad

def _complete(seed_dir):
    """Entraînement réellement terminé ? (results.csv : 150 époques, ou arrêt anticipé par la patience)"""
    p = os.path.join(seed_dir, 'results.csv')
    if not os.path.exists(p): return False, 0, -1
    df = pd.read_csv(p); df.columns = df.columns.str.strip()
    n = len(df)
    if 'metrics/mAP50(B)' in df.columns and 'metrics/mAP50-95(B)' in df.columns:
        fit = 0.1 * df['metrics/mAP50(B)'] + 0.9 * df['metrics/mAP50-95(B)']      # fitness ultralytics
        best = int(fit.idxmax())
    else:
        best = -1
    return (n >= EPOCHS or (best >= 0 and (n - 1 - best) >= PATIENCE)), n, best

print(f"{'seed':<6}{'nbs':<6}{'batch':<7}{'optim.':<8}{'lr0':<9}{'époques':<9}{'meilleure':<10}statut")
_args = {}
for _s in RUN_SEEDS:
    _d = os.path.join(MODEL_DIR, f'seed_{_s}')
    _p = os.path.join(_d, 'args.yaml')
    assert os.path.exists(_p), f'args.yaml introuvable pour la graine {_s} : {_p}'
    _a = yaml.safe_load(open(_p)); _args[_s] = _a
    _ok, _n, _b = _complete(_d)
    print(f"{_s:<6}{str(_a.get('nbs')):<6}{str(_a.get('batch')):<7}{str(_a.get('optimizer')):<8}{str(_a.get('lr0')):<9}"
          f"{_n:<9}{_b + 1:<10}{'terminé' if _ok else 'INCOMPLET'}")
    _bad = _check_args(_a, EXPECTED_ARGS)
    assert not _bad, f'Graine {_s} : réglages différents de ceux du tableau 1 →\n  ' + '\n  '.join(_bad)
    assert _ok, f'Graine {_s} : entraînement incomplet ({_n} époques) : relancer la Cellule 5 pour cette graine.'
print(f'✅ nbs={EXPECTED_NBS}, SGD, lr0={LR0}, batch=16, entraînements terminés : conformes au tableau 1 pour les {len(RUN_SEEDS)} graines.\n')

detect_by_seed = {}
for seed in RUN_SEEDS:
    out_path = os.path.join(MODEL_DIR, f'metriques_detection_seed{seed}.json')
    if os.path.exists(out_path):
        with open(out_path) as f:
            detect_by_seed[seed] = json.load(f)
        print(f'✅ Seed {seed} : métriques de détection déjà calculées, chargées.')
        continue

    print(f'\n📐 Seed {seed} — précision / rappel / F1...')
    weights = os.path.join(MODEL_DIR, f'seed_{seed}', 'weights', 'best.pt')
    assert os.path.exists(weights), f'Poids introuvables (seed {seed}) : {weights}'
    model = YOLO(weights)

    # 1) seuil de confiance qui maximise le F1 (boîtes) sur la validation
    val_data = _collect(model, 'val')
    f1_val = np.array([_counts(val_data, t)['F1'] for t in THR_GRID])
    thr = float(THR_GRID[int(np.argmax(f1_val))])

    # 2) métriques sur le test à ce seuil
    test_data = _collect(model, 'test')
    m = _counts(test_data, thr)

    # 3) mAP@50 et mAP@50-95 (même appel que la Cellule 6, iou=0.5 = NMS)
    vr = model.val(data=data_yaml_path, split='test', imgsz=IMG_SIZE,
                   iou=0.5, conf=0.001, batch=16, verbose=False, plots=False)
    map50, map5095 = float(vr.box.map50), float(vr.box.map)

    # Contrôle de cohérence : le mAP@50 recalculé doit retrouver celui déjà enregistré
    prev = os.path.join(MODEL_DIR, f'metriques_seed{seed}.json')
    if os.path.exists(prev):
        with open(prev) as f:
            _d = json.load(f)
            prev_map50 = _d['Fracture']['mAP50'] if 'Fracture' in _d else _d['mAP50']
        if abs(prev_map50 - map50) > 0.002:
            print(f'   ⚠️ mAP50 recalculé ({map50:.4f}) ≠ mAP50 enregistré ({prev_map50:.4f}) : '
                  f'vérifier que ce sont bien les mêmes poids.')
    if map50 < 0.1:
        print("   ⚠️ mAP50 anormalement bas : checkpoint probablement issu d'un entraînement interrompu.")

    res = {'seuil_F1_val': thr, 'F1_val_max': round(float(f1_val.max()), 4),
           'mAP50': round(map50, 4), 'mAP50_95': round(map5095, 4),
           'nbs': _args[seed].get('nbs'), 'batch': _args[seed].get('batch')}
    res.update({k: (round(v, 4) if isinstance(v, float) else v) for k, v in m.items()})
    with open(out_path, 'w') as f:
        json.dump(res, f, indent=2)
    detect_by_seed[seed] = res
    print(f"   seuil={thr:.3f} | P={res['Precision']:.3f} R={res['Rappel']:.3f} F1={res['F1']:.3f} "
          f"| FP/img normale={res['FP_par_image_normale']:.3f} | mAP50-95={map5095:.3f}")

    del model; gc.collect()
    if torch.cuda.is_available():
        torch.cuda.empty_cache()

# ── Agrégation : moyenne ± écart-type (ddof=1, correction de Bessel, n = nb de graines) ──
keys = ['mAP50', 'Precision', 'Rappel', 'F1', 'mAP50_95', 'TP', 'FP', 'FN', 'FP_par_image_normale', 'Specificite_image', 'seuil_F1_val']
agg_det = {'n_seeds': len(RUN_SEEDS), 'poids': 'YOLOv11n (nbs=%d)' % EXPECTED_NBS, 'ddof': 1}
print('\nYOLOv11n — détection (boîtes, IoU >= 0.5), moyenne ± é.-t. (ddof=1) sur', len(RUN_SEEDS), 'graines')
for k in keys:
    v = np.array([detect_by_seed[s][k] for s in RUN_SEEDS], dtype=float)
    agg_det[k] = {'mean': round(float(v.mean()), 4), 'std': round(float(v.std(ddof=1)), 4)}
    print(f"  {k:<22s} {v.mean():.3f} ± {v.std(ddof=1):.3f}")
with open(os.path.join(MODEL_DIR, 'resultats_detection_agreges.json'), 'w') as f:
    json.dump(agg_det, f, indent=2)
